In [59]:
import ast
import json
import pandas as pd
import numpy as np

In [60]:
def parse_json(valor):
    """Normalizes the contents of the datasets column into a list of dicts.
    Accepts: a list/tuple/ndarray of dicts, a single dict, a JSON string, 
    a string in Python literal format, and null/empty values (returns []).
    """
    if isinstance(valor, dict):
        return [valor]
    if isinstance(valor, (list, tuple)):
        return list(valor)
    if hasattr(valor, "tolist"):        
        return valor.tolist()
    if valor is None or pd.isna(valor):
        return []

    # String
    texto = str(valor).strip()
    if texto in ("", "[]", "nan", "None"):
        return []
    try:
        resultado = json.loads(texto)
    except json.JSONDecodeError:
        resultado = ast.literal_eval(texto)

    return [resultado] if isinstance(resultado, dict) else list(resultado)

In [61]:
# Lê os dados:
with open('../dados/limpos/usecases_current.json', 'r') as f:
    data = json.load(f)
usecases_raw = data['data']

In [62]:
# Seleciona casos considerados de uso de dados públicos:
usecases = list(filter(lambda uc: (uc['status_published'] == True) or ((fillnone(uc['comment'], '').find('Ocultado porque') == -1) and (fillnone(uc['comment'], '').find('não atendem aos nossos critérios') == -1)) , usecases_raw))

In [63]:
# Converte os casos para dataframe
df = pd.DataFrame(usecases)

In [64]:
# Amostra
df.sample(3)

,hash_id,name,url,url_archive,description,pub_date,authors,authors_id,geo_level,countries,...,url_image,comment,datasets,record_date,modified_date,status_published,status_review,type_es,topics_es,countries_es
548,2437576788,Justiça expõe nomes de mulheres e crianças vít...,https://g1.globo.com/politica/noticia/2025/06/...,http://archive.today/WYcT5,A reportagem revelou que 120 mulheres e crianç...,06/2025,[Judite Cypreste],[None],Países,[Brasil],...,https://raw.githubusercontent.com/cewebbr/cord...,Cadastrado por Jacqueline (Ceweb.br).,"[{'data_name': 'Mandados de Prisão', 'data_ins...",2025-10-17,None,True,False,[artículo periodístico],"[Defensa y Seguridad, Justicia y Legislación, ...",[Brasil]
257,2179656596,Características dos domicílios - Infográfico,https://dados.al.gov.br/catalogo/dataset/carac...,https://archive.ph/1LGul,Infográfico com os principais dados acerca das...,03/2024,"[Secretaria de Estado do Planejamento, Gestão ...","[None, None]",Unidades federativas,[Brasil],...,https://raw.githubusercontent.com/cewebbr/cord...,Cadastrado por César Calafrioli.,[{'data_name': 'Características dos domicílios...,2026-03-05,2026-03-18,True,False,[panel de control o infografía],"[Vivienda, Saneamiento y Urbanismo]",[Brasil]
114,1359663761,Siga o dinheiro,https://www.sigaodinheiro.org/dados,https://,O Siga o Dinheiro transforma as informações br...,01/2022,"[JOTA, basedosdados - bd]","[None, None]",Países,[Brasil],...,https://raw.githubusercontent.com/cewebbr/cord...,Cadastrado por César Calafrioli.,"[{'data_name': 'Eleições Brasileiras', 'data_i...",2026-05-03,2026-05-03,True,False,[panel de control o infografía],[Gobierno y Política],[Brasil]


## Cria datafame de datasets

In [65]:
# Converte a coluna "datasets" em um dataframe
registros = []
for hash_id, valor in zip(df["hash_id"], df["datasets"]):
    for item in parse_json(valor):
        registros.append({"hash_id": hash_id, **item})

df_datasets = pd.DataFrame(registros)

assert df_datasets["hash_id"].isin(df["hash_id"]).all()

In [66]:
df_datasets.sample(3)

,hash_id,data_name,data_institution,data_url,data_license,data_format,data_periodical
1369,2410563585,Índice de Desenvolvimento da Educação Básica -...,Instituto Nacional de Estudos e Pesquisas Educ...,https://dados.gov.br/dados/conjuntos-dados/ine...,CC-BY,"[ZIP, ODS, XLSX, TXT]",False
946,4037909796,Equipes de Saúde da Família (ESF),Ministério da Saúde - MS,http://tabnet.datasus.gov.br/cgi/deftohtm.exe?...,Inexistente,[CSV],False
237,498318146,Banco de dados das cheias na Região Hidrográfi...,Universidade Federal do Rio Grande do Sul - UFRGS,https://zenodo.org/doi/10.5281/zenodo.11164049,CC-BY,[GPKG],False


In [67]:
df_datasets.shape

(1868, 7)

### Identifica conjuntos de dados sem licença

In [68]:
# Verifica existência de casos com "data_license" em branco
df_datasets['data_license'].isna().any()

True

In [69]:
df_datasets['data_license'].unique()

array(['Inexistente', 'CC-BY', 'Outra', 'CC0', 'CC BY-SA', 'CC BY-ND',
       'CC BY-NC', 'CC BY-NC-ND', 'CC BY-NC-SA', 'ODbL', 'GPL', 'AGPL',
       None], dtype=object)

In [83]:
df_datasets_without_license = df_datasets[df_datasets['data_license'].isna()]

In [84]:
df_datasets_without_license.shape

(14, 7)

In [85]:
df_r = df[['hash_id', 'name']]

In [86]:
df_datasets_without_license = df_datasets_without_license.merge(df_r, on='hash_id')

In [88]:
df_datasets_without_license.sample(3)

,hash_id,data_name,data_institution,data_url,data_license,data_format,data_periodical,name
3,4184481763,Desoxinivalenol - 3D Conformer,National Center for Biotechnology Information ...,https://pubchem.ncbi.nlm.nih.gov/compound/40024,None,"[XML, JSON, PNG, Outro]",False,Estudo teórico-computacional da interação de m...
0,401574645,Votação Nominal,Senado Federal - SF,https://www12.senado.leg.br/dados-abertos/conj...,None,"[JSON, XML, API]",False,Senado aberto: dados abertos das votações nomi...
9,1784118771,Proprietários de escravizados,Pontifícia Universidade Católica do Rio de Jan...,https://www.opodereaescravidao.com/banco-de-dados,None,[],False,Escravizador na pele de abolicionista: aboliçã...


### Quantifica licenças

In [28]:
# Contagem geral por tipo de licença 
lista = df_datasets['data_license'].value_counts(ascending=False)
lista

Inexistente    913
CC-BY          539
CC BY-ND       103
CC0             76
Outra           68
ODbL            56
CC BY-SA        47
CC BY-NC        18
CC BY-NC-ND     13
GPL             13
CC BY-NC-SA      6
AGPL             2
Name: data_license, dtype: int64

In [41]:
# Proprietárias de conjuntos de dados (amostra)
df_datasets_licenca_inexistente = df_datasets[df_datasets['data_license'] == 'Inexistente']

In [49]:
# Soma dos dataframes das top 15 principais proprietárias de dados com licença inexistente
df_datasets_licenca_inexistente['data_institution'].value_counts(ascending=False)

Instituto Brasileiro de Geografia e Estatística - IBGE                                               209
Ministério da Saúde - MS                                                                             105
National Center for Biotechnology Information - NCBI                                                  58
National Institutes of Health - NIH                                                                   33
Yahoo Finance                                                                                         31
                                                                                                    ... 
Ministério da Justiça e Segurança Pública - MJSP                                                       1
Secretaria da Administração Penitenciári e Ressocialização do Governo do Estado do Ceará - SAP/CE      1
World Justice Project                                                                                  1
Instituto Brasileiro de Geografia e Estatística - IBGE;

## Solução proposta:

1. Preenchimento de dados faltantes
1.Criação de uma categoria "licença proprietária"
1. Parsear url dos conjuntos de dados
1. Identificar padrão no prefixo das urls
1. Pesquisar licença do portal
1. Atribuir licenças via script, caso possível